# 00 — Load & preprocess the AA data (team 07)

**Owner:** Patrick · **Run this first.** Every other notebook in this folder starts with
`%run ./00_load_and_preprocess.ipynb`, so they all see the exact same cleaned data.

What this notebook does:
1. Finds `PAX_FORECASTING` and `AIRPORT_INFO` in the repo's git-ignored `data/raw/` folder.
2. Loads them into pandas.
3. Applies **mechanical** cleaning only — whitespace, number/date types, exact duplicate rows.

What it deliberately does **not** do: impute missing values, drop columns, remove outliers, or
engineer features. Those are EDA decisions (notebooks 01–04) and feed the feature set (05).

> **NDA reminder.** This is real American Airlines data. Never commit it, never paste rows or
> screenshots of it anywhere outside the team, and **clear outputs before every commit**
> (Kernel → Restart & Clear Output) so no data ends up in the notebook file.

## 1. Paths and settings

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start=None):
    """Walk up from the current folder until we hit the repo root (has DATA.md and .git)."""
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "DATA.md").exists() and (p / ".git").exists():
            return p
    raise FileNotFoundError("Couldn't find the repo root — open this notebook from inside your clone.")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "raw"
MODELING_DIR = REPO_ROOT / "team-07-modeling"
RESULTS_DIR = MODELING_DIR / "results"
MODELS_DIR = MODELING_DIR / "models"

TARGET = "OD_PAX"  # origin-destination passengers; confirm the definition in Snowflake

# Set to e.g. 200_000 for a quick first pass if the full file is slow on your laptop.
# Leave as None for real results.
NROWS = None

print("repo root:", REPO_ROOT)
print("raw data folder:", RAW_DIR)

## 2. Find and load the two files

In [ ]:
DATA_SUFFIXES = (".csv", ".csv.gz", ".parquet", ".xlsx")


def find_source(stem):
    """Return the one file in data/raw/ whose name contains `stem` (case-insensitive)."""
    if not RAW_DIR.exists():
        raise FileNotFoundError(f"{RAW_DIR} doesn't exist. Create it and put the Drive files in it "
                                "(see team-07-modeling/README.md).")
    matches = sorted(p for p in RAW_DIR.iterdir()
                     if p.is_file() and stem.lower() in p.name.lower()
                     and p.name.lower().endswith(DATA_SUFFIXES))
    if not matches:
        raise FileNotFoundError(f"No file with '{stem}' in its name in {RAW_DIR}. "
                                "Download it from the team Drive link (ask Patrick).")
    if len(matches) > 1:
        raise ValueError(f"More than one file matches '{stem}': {[m.name for m in matches]}. "
                         "Keep exactly one copy in data/raw/.")
    return matches[0]


import re

ID_WORDS = {"ID", "CODE", "CD", "ZIP", "FIPS", "IATA", "ICAO", "KEY"}
DATE_WORDS = {"DATE", "MONTH", "PERIOD", "DT", "TIME", "YYYYMM"}


def name_has(col, words):
    """True if any underscore/space-separated word of the column name is in `words` (ORIGIN_AIRPORT_ID -> ID)."""
    return any(w in words for w in re.split(r"[^A-Z0-9]+", str(col).upper()))


def read_any(path, nrows=None):
    name = path.name.lower()
    if name.endswith((".csv", ".csv.gz")):
        header = pd.read_csv(path, nrows=0).columns
        id_cols = {c: str for c in header if name_has(c, ID_WORDS)}  # keep leading zeros
        return pd.read_csv(path, nrows=nrows, low_memory=False, dtype=id_cols)
    if name.endswith(".parquet"):  # needs pyarrow
        df = pd.read_parquet(path)
        return df.head(nrows) if nrows else df
    if name.endswith(".xlsx"):  # needs openpyxl
        return pd.read_excel(path, nrows=nrows)
    raise ValueError(f"Don't know how to read {path.name}")


pax_path = find_source("PAX_FORECASTING")
airport_path = find_source("AIRPORT_INFO")

pax_raw = read_any(pax_path, nrows=NROWS)
airports_raw = read_any(airport_path)

print(f"{pax_path.name}: {pax_raw.shape[0]:,} rows x {pax_raw.shape[1]} cols")
print(f"{airport_path.name}: {airports_raw.shape[0]:,} rows x {airports_raw.shape[1]} cols")

## 3. Mechanical cleaning

Each step prints what it changed, so the cleaning is auditable. If a step converts a column it
shouldn't have (e.g. an ID that should stay text), add it to `KEEP_AS_TEXT` and re-run.

| Step | Why |
|---|---|
| Strip whitespace from column names and text values; `""` → missing | Exports often pad strings; `"DFW "` and `"DFW"` must be the same airport |
| Numeric-looking text → numbers (`"1,234"` → `1234`) | So `describe()`, plots and models treat them as numbers |
| Date-like text columns → `datetime` | So the time axis can be sorted, grouped by month, and split by date (no random splits on time data) |
| Drop exact duplicate rows | An identical row twice is an export artifact, not two observations; the count is printed so 01 can check it |

In [ ]:
# ID-like columns (ID_WORDS, section 2) are never converted to numbers; DATE_WORDS columns are tried as dates.
KEEP_AS_TEXT = []  # add column names here if a conversion below was wrong


def text_columns(df):
    # pandas 2 stores text as "object", pandas 3 as "str"/"string" — handle both
    return list(df.select_dtypes(include=["object", "string"]).columns)


def preprocess(df, name):
    df = df.copy()
    report = {"dataset": name, "rows_in": len(df)}

    # 1. whitespace
    df.columns = [str(c).strip() for c in df.columns]
    for c in text_columns(df):
        df[c] = df[c].map(lambda v: v.strip() if isinstance(v, str) else v).replace("", np.nan)

    # 2. numeric-looking text -> numbers
    to_number = []
    for c in text_columns(df):
        if c in KEEP_AS_TEXT or name_has(c, ID_WORDS):
            continue
        present = df[c].dropna().astype(str).str.replace(",", "", regex=False)
        if present.empty:
            continue
        parsed = pd.to_numeric(present, errors="coerce")
        if parsed.notna().mean() >= 0.99:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "", regex=False), errors="coerce")
            to_number.append(c)
    report["text_to_number"] = to_number

    # 3. date-like text -> datetime
    to_date = []
    for c in text_columns(df):
        if c in KEEP_AS_TEXT or not name_has(c, DATE_WORDS):
            continue
        parsed = pd.to_datetime(df[c], errors="coerce")
        if df[c].notna().sum() and parsed.notna().sum() >= 0.95 * df[c].notna().sum():
            df[c] = parsed
            to_date.append(c)
    report["text_to_date"] = to_date

    # 4. exact duplicate rows
    dup = int(df.duplicated().sum())
    df = df.drop_duplicates().reset_index(drop=True)
    report["duplicate_rows_dropped"] = dup
    report["rows_out"] = len(df)

    for k, v in report.items():
        print(f"  {k}: {v}")
    return df


print("PAX_FORECASTING")
pax = preprocess(pax_raw, "PAX_FORECASTING")
print("\nAIRPORT_INFO")
airports = preprocess(airports_raw, "AIRPORT_INFO")

## 4. Sanity checks

In [ ]:
if TARGET not in pax.columns:
    print(f"WARNING: target column {TARGET!r} not found. Columns are:\n{list(pax.columns)}\n"
          "Update TARGET in section 1 once the team confirms the right column in Snowflake.")
else:
    print(f"target {TARGET!r}: dtype={pax[TARGET].dtype}, missing={pax[TARGET].isna().sum():,}")

print("\nPAX_FORECASTING column types:")
print(pax.dtypes.value_counts().to_string())
print("\nAIRPORT_INFO column types:")
print(airports.dtypes.value_counts().to_string())

## 5. What you have now

| Name | What it is |
|---|---|
| `pax` | cleaned `PAX_FORECASTING` — the main table |
| `airports` | cleaned `AIRPORT_INFO` — extra airport attributes, not joined yet (04 decides how) |
| `pax_raw`, `airports_raw` | untouched copies, for checking what cleaning changed |
| `TARGET` | the column we're forecasting |
| `RESULTS_DIR`, `MODELS_DIR` | where to save plots/tables and models |

**Using this in your notebook:** put `%run ./00_load_and_preprocess.ipynb` in your first code cell.
Don't edit this notebook on your own branch — if the cleaning needs to change, tell Patrick so
everyone stays on the same data.